# ASR Simulation 3: Grid 2 (10ft near well) with Chemistry from MF6RTM Example 5 (Appelo 1998) 

This simulation adds **arsenic redox chemistry** -- from Wallis et al 2011 -- to the 3D transport models of the simple ASR test case.

For information and exploration of the simple ASR Modflow 6 simulation used throughout this repository, see `sims/sim00-mf6only/mf6_explore.ipynb`.

The workflow for this example:
- Read geochemical components and their initial and boundary concentrations from PHREEQC input files
- Create new Modflow 6 transport model for each aqueous phase (components in the Solution blocks) and add their initial concentrations over the entire DISV grid.
- Modify the Modflow 6 Flow Well package Stress Period Data (SPD) by adding Solution component concentrations.
- Run the modified Modflow 6 for conservative transport of all components (i.e. no coupling to PHREEQC)
- Run the coupled Modflow 6 & PHREEQC models for the entire simulation

NOTE: This [Jupytext](https://jupytext.readthedocs.io/en/latest/index.html) paired notebook, with paired `.py` and `.ipynb` files. 
- If using VS Code, install the the [Jupytext Sync extension](https://jupytext.readthedocs.io/en/latest/vs-code.html) for maximum benefit.

# Installation and Setup

Create a custom conda virtual environment can be created using the `environment.yml` file included in this repo. 

Complet the setup by running the `01-GettingStarted.ipynb` notebook.

## Python Imports

In [1]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import flopy
from modflowapi import ModflowApi

In [2]:
# Import the MFRTM package, installed using `conda develop`
import mf6rtm

display(mf6rtm.__file__)
try:
    # if current LimnoTech development version
    display(mf6rtm.__version__)
except AttributeError:
    pass

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


'/Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm/mf6rtm/__init__.py'

'0.3.1+develop'

In [3]:
import utils # from this repo

### If you get `ModuleNotFoundError`

Run the `01-GettingStarted.ipynb` notebook to install `mf6rtm` using `conda develop`.

## Set Paths to Input and Output Files with `pathlib`

Use the [pathlib](https://docs.python.org/3/library/pathlib.html) library 
(built-in to Python 3) to manage paths indpendentely of OS or environment. 
See this [blog post](https://medium.com/@ageitgey/python-3-quick-tip-the-easy-way-to-deal-with-file-paths-on-windows-mac-and-linux-11a072b58d5f) 
to learn about the many benefits over using the `os` library.

In [4]:
# Find your current working directory, which should be folder for this notebook.
working_dir = Path.cwd()
# Find repository path (i.e. the parent to `/examples` directory for this notebook)
repo_path = working_dir.parent.parent
repo_path

PosixPath('/Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm-asr-example')

In [5]:
# Simulation based on chemical inputs
# simulation_name = working_dir.name
simulation_name = "MF6RTM Example 5"
simulation_name

'MF6RTM Example 5'

In [6]:
# Path to simulation workspace, which is git-ignored and 
# will get over-written with each run of this notebook
sim_ws = working_dir / 'ws2x' # Grid 2
sim_ws.mkdir(parents=True, exist_ok=True)

### Reset Workspace

In [7]:
# Delete previous contents from simulation
if sim_ws.exists():
    try:
        shutil.rmtree(sim_ws)
        print(f"Directory '{sim_ws}' and its contents removed successfully.")
        sim_ws.mkdir(parents=True, exist_ok=True)
    except OSError as e:
        print(f"Error: {sim_ws} : {e.strerror}")
else:
    print(f"Directory '{sim_ws}' does not exist.")

Directory '/Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm-asr-example/sims/sim03-Wallis2011/ws2x' and its contents removed successfully.


### Modflow Inputs

In [8]:
# Modflow inputs file folder
# Grid 2 = 10ft resolution near well (vs 2ft for original)
mf6_inputs_path = repo_path / 'data' / 'MF6_ASR_DISV_inputs2' # Grid 2

In [9]:
# Copy input files to simulation workspace directory)
shutil.copytree(mf6_inputs_path, sim_ws, dirs_exist_ok=True)

PosixPath('/Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm-asr-example/sims/sim03-Wallis2011/ws2x')

In [10]:
# Add required empty output folders
folders = [
    'flow_output',
    'trans-TDS_output',
    'trans-temp_output',
]
for folder in folders:
    path = sim_ws / folder
    path.mkdir(parents=True, exist_ok=True)

In [11]:
# Set filepath for the MF6 simulation configuration file
sim_nam_file_path = sim_ws / "mfsim.nam"
assert sim_nam_file_path.exists()
sim_nam_file_path

PosixPath('/Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm-asr-example/sims/sim03-Wallis2011/ws2x/mfsim.nam')

In [12]:
# Set modflow input path equal to simulation workspace
mf6_input_path = sim_ws

### PHREEQC Inputs

In [13]:
# Phreeqc input file folder
chem_inputs_path = working_dir / "chem_inputs"

chem_prefix = "ex5_"
chem_input_files_match = chem_inputs_path.glob(f"{chem_prefix}*")
chem_input_files = [file.name for file in chem_input_files_match]
chem_input_files

['ex5_exchanges.csv',
 'ex5_surfaces.csv',
 'ex5_kinetic_phases.csv',
 'ex5_equilibrium_phases.csv',
 'ex5_postfix.phqr',
 'ex5_solutions.csv']

In [14]:
# Copy input files to simulation workspace directory (i.e. project path)
for file in chem_input_files:
    shutil.copy2(chem_inputs_path / file, sim_ws)

In [15]:
# Path to PHREEQC Block Input CSV Files
solutions_filepath = sim_ws / f"{chem_prefix}solutions.csv"
exchanges_filepath = sim_ws / f"{chem_prefix}exchanges.csv"
equilibrium_phases_filepath = sim_ws / f"{chem_prefix}equilibrium_phases.csv"
surfaces_filepath = sim_ws / f"{chem_prefix}surfaces.csv"
kinetic_phases_filepath = sim_ws / f"{chem_prefix}kinetic_phases.csv"

assert solutions_filepath.exists() and exchanges_filepath.exists()
assert equilibrium_phases_filepath.exists() and surfaces_filepath.exists()
assert kinetic_phases_filepath.exists()

In [16]:
# Path to file with PHREEQC Input "postfix" instructions
# to be appended to the PHREEQC Input file (*.pqi) created by mf6rtm
postfix_filepath = sim_ws /  f"{chem_prefix}postfix.phqr"
assert postfix_filepath.exists()

In [17]:
# Select PHREEQC database file
# phreeqc_database_file = "datab.dat"
phreeqc_database_file = "phreeqc.dat" # used in Ex5 & 6?
# phreeqc_database_file = 'pht3d_datab.dat' # used in Ex4
phreeqc_databases_path = repo_path / "data" / "chem_databases"
phreeqc_database_filepath = phreeqc_databases_path / phreeqc_database_file
assert phreeqc_database_filepath.exists(), "PHREEQC database file missing"

In [18]:
# Paths to PHREEQC configuration files that will be created by mf6trm
# PHREEQC Input file (*.pqi)
phreeqc_input_filepath = sim_ws / "phinp.dat"
# PhreeqcRM YAML config file
phreeqcrm_yaml_filepath = sim_ws / "mf6rtm.yaml"

## Set Path to MF6 Executable & Library
Different versions can be downloaded from: https://github.com/MODFLOW-ORG/executables to a folder similar to this: `bin/mf6.5.0/macarm` 

On Mac, will need to give permissions with these terminal commands from the 
```sh
xattr -dr com.apple.quarantine mf6
xattr -dr com.apple.quarantine libmf6.dylib
```


In [19]:
use_version_installed_with_modflowapi = False
# user = "Laren"
user = "Anthony"
os = "macarm"

# version = "6.4.2"
# version = "6.5.0"
version = "6.7.0"

try:
    mf6_exe = Path(flopy.which("mf6"))
    dll = mf6_exe.parent.parent / "lib" / "libmf6.dylib" # MacOS only for now
    mf6_version = !{mf6_exe} --version
    mf6dll_version = ModflowApi(dll).get_version()
    print(f"Executable & library installed with modflowapi: {mf6_version[1]}, dll: {mf6dll_version}")
except Exception:
    print("Modflow executables not found in environment")

if use_version_installed_with_modflowapi:
    print(f"Using executable installed with modflowapi: {mf6_version[1]}")
else:
    if user == "Lauren":
        # If using executable from GMS
        mf6_bin_path = Path(r"C:/program files/gms 10.8 64-bit/python/lib/site-packages/xms/executables/modflow6")
        mf6_exe = mf6_bin_path / "mf6.exe"
        dll = mf6_bin_path / "libmf6.dll"
    elif user == "Anthony":
        mf6_potential_paths = [
            repo_path / "bin" / f"mf{version}" / os,
            repo_path / "bin" / f"mf{version}_{os}" / "bin",
        ]
        for path in mf6_potential_paths:
            if path.exists():
                mf6_bin_path = path
        mf6_exe = mf6_bin_path / "mf6"
        dll = mf6_bin_path / "libmf6.dylib"
    else:
        print("Create a new user and set paths to mf6 and libmf6")
    mf6_version = !{mf6_exe} --version
    mf6dll_version = ModflowApi(dll).get_version()
    print(f"User-selected executable ({mf6_exe.exists()}): {mf6_version[1]}, dll: {mf6dll_version}")

Executable & library installed with modflowapi: mf6: 6.7.0 02/06/2026, dll: 6.7.0
User-selected executable (True): mf6: 6.7.0 02/05/2026, dll: 6.7.0


In [20]:
# Copy executable and library to simulation workspace
shutil.copy2(mf6_exe, sim_ws)
shutil.copy2(dll, sim_ws)
(sim_ws/mf6_exe.name).exists()

True

# Load Modflow 6 Simulation for Reactive Transport
To set up PhreeqcRM and MF6RTM simulation objects.
Modifies Modflow 6 input files to include transport models for all reactive transport species.

For addtional information and exploration of the simple ASR Modflow 6 simulation used throughout this repository, see `sims/sim00-mf6only/mf6_explore.ipynb`

In [21]:
# Load simulation using Flopy
sim = flopy.mf6.MFSimulation.load(
    sim_ws=sim_ws,
    exe_name=mf6_exe,  #'mf6',
    verbosity_level=0,
)
sim.model_names

['flow', 'trans-tds', 'trans-temp']

## Modify Flow Model

In [22]:
# load existing gwf model
for model_name in sim.model_names:
    model = sim.get_model(model_name)
    if model.model_type == "gwf6":
        gwf = model

# removes buy package from gwf model
gwf.remove_package("buy")
gwf.get_package_list()

['DISV', 'NPF', 'IC', 'OC', 'STO', 'WEL', 'CHD', 'GHB']

In [23]:
# modify output control package to not print head to .lst file
oc = gwf.get_package("oc")
print_record = oc.printrecord.get_data()
print_rec = print_record[0]
mask = ~(
    (print_rec.rtype == "head")
    & (print_rec.ocsetting == "all")
    & (print_rec.ocsetting_data == None)
)
print_record_new = {}
print_record_new[0] = print_record[0][mask]
oc.printrecord.set_data(print_record_new)

In [24]:
# modify npf package to save specific discharge
npf = gwf.get_package("npf")
npf.save_specific_discharge = True

## Read Grid Info

In [25]:
# Get groundwater model names and grid info
for model_name in sim.model_names:
    # Collect model info 
    model = sim.get_model(model_name)
    model_type = model.model_type
    grid_type = model.get_grid_type()
    grid_units = model.modelgrid.units
    # Collect grid information
    grid_package = model.get_package(grid_type.name)
    nlay = grid_package.nlay.get_data()  # number of layers
    ncpl = grid_package.ncpl.get_data()  # number of cells per layer
    print(f"{model_name}: ", model_type, grid_type.name, grid_units, nlay, ncpl)

flow:  gwf6 DISV feet 5 957
trans-tds:  gwt6 DISV feet 5 957
trans-temp:  gwt6 DISV feet 5 957


In [26]:
# Use spatial discretization info from the last model
# Calculate total number of grid cells
nxyz = nlay * ncpl
nxyz

4785

In [27]:
# lookup cell ID of wel package cell
wel_spd = gwf.wel.stress_period_data.array
wel_cellid = wel_spd[0]["cellid"][0]
display(wel_cellid)

wel_lay = wel_cellid[0]
wel_cellnum = wel_cellid[1]

(2, 462)

### Cell Spacing

In [28]:
grid_package.length_units

{internal}
('feet')

In [29]:
celldata = grid_package.cell2d.get_data()
# data stored in numpy record arrays  
# can be easily converted to pandas dataframes
cells_df = pd.DataFrame.from_records(celldata, index='icell2d')
cells_df

,xc,yc,ncvert,icvert_0,icvert_1,icvert_2,icvert_3,icvert_4,icvert_5
icell2d,,,,,,,,,
0,694376.127565,1.027689e+06,4,3,2,1,0,None,None
1,694530.764694,1.027689e+06,4,5,4,2,3,None,None
2,694685.401823,1.027689e+06,4,7,6,4,5,None,None
3,694840.038952,1.027689e+06,4,9,8,6,7,None,None
4,694994.676081,1.027689e+06,4,11,10,8,9,None,None
...,...,...,...,...,...,...,...,...,...
952,698396.692919,1.023169e+06,4,993,1025,1024,992,None,None
953,698551.330048,1.023169e+06,4,994,1026,1025,993,None,None
954,698705.967177,1.023169e+06,4,995,1027,1026,994,None,None


### Cell Volumes

In [30]:
### Calculate grid cell volume
cell2D = grid_package.cell2d.get_data()
cell2D_df = pd.DataFrame.from_records(cell2D, index='icell2d')
vertices = grid_package.vertices.get_data()
vertices_df = pd.DataFrame.from_records(vertices, index='iv')
top = grid_package.top.array
botm = grid_package.botm.array

### Calculate surface area of each grid cell within a single layer
for cell in range(len(cell2D_df)):
    ### get vertice coordinates to calc surface area
    temp_cell_info = cell2D_df.iloc[[cell]]
    # read each vert_1 - 5
    icverts = temp_cell_info.filter(like="icvert_").iloc[0].to_list()
    # remove Nones
    icverts_clean = [int(v) for v in icverts if v is not None]
    # look up (x,y) and create x and y arrays
    x_l = vertices_df.loc[icverts_clean,"xv"].to_numpy()
    y_l = vertices_df.loc[icverts_clean,"yv"].to_numpy()
    # calculate surface area based on min/max x/y from array
    surface_area = (np.max(x_l)-np.min(x_l)) * (np.max(y_l) - np.min(y_l))
    cell2D_df.loc[cell,'surface_area'] = surface_area

### Calc layer thickness for each layer
# initialize thickness array
thickness = np.zeros_like(botm)
# for layer 1:
thickness[0] = top - botm[0]
# for layers 2:nlay
thickness[1:] = botm[:-1] - botm[1:]

### Calculate volume for each grid cell
# initialize volume array
cell_volumes = np.zeros((nlay,ncpl))
# calculate volume for entire grid
for k in range(nlay):
    cell_volumes[k,:] = cell2D_df['surface_area'].to_numpy() * thickness[k,:]

In [31]:
# volume of cells near well screen
cell_volumes[2, wel_cellnum-6:wel_cellnum+6]

array([ 352455.27301523,  352455.27301523,  352455.27301524,
        352455.27301418,   88113.81825434,   88113.81825381,
         88113.81825434,   88113.81825381,  352455.27301523,
       1409821.09205884, 1409821.09205884, 1409821.09206095])

In [32]:
# TODO: Get flat Cell Index to (cellid_layer, cellid_cell) mapping
# for exploring phreeqcrm outputs

In [33]:
cell_flat_index = np.array(range(nlay*ncpl))
cell_flat_index

array([   0,    1,    2, ..., 4782, 4783, 4784], shape=(4785,))

In [34]:
cellid_flatmap = np.reshape(cell_flat_index, (nlay,ncpl))
cellid_flatmap

array([[   0,    1,    2, ...,  954,  955,  956],
       [ 957,  958,  959, ..., 1911, 1912, 1913],
       [1914, 1915, 1916, ..., 2868, 2869, 2870],
       [2871, 2872, 2873, ..., 3825, 3826, 3827],
       [3828, 3829, 3830, ..., 4782, 4783, 4784]], shape=(5, 957))

In [35]:
cellid_layer = wel_lay
cellid_cell = wel_cellnum
cellid_flatmap[cellid_layer, cellid_cell]

np.int64(2376)

## Read Time Info

### Time Steps

In [36]:
# Get time discretization info from the `tdis` package
tdis = sim.tdis
nper = tdis.nper.get_data()          # number of stress periods
perioddata = tdis.perioddata.get_data() # record array
nstp = perioddata['nstp']            # number of timesteps per stress period
perlen = perioddata['perlen']        # length of stress periods
tsmult = perioddata['tsmult']        # timestep multiplier
t_units = tdis.time_units.get_data() # units

print(f'{nper} stress periods. Units: {t_units}')
perioddata

21 stress periods. Units: days


rec.array([( 1., 1, 1.), (13., 2, 1.), (28., 3, 1.), (31., 3, 1.),
           (30., 3, 1.), (31., 3, 1.), (30., 3, 1.), ( 9., 1, 1.),
           (22., 2, 1.), (31., 3, 1.), (30., 3, 1.), (31., 3, 1.),
           (30., 3, 1.), (31., 3, 1.), ( 3., 1, 1.), (28., 3, 1.),
           (28., 3, 1.), (31., 3, 1.), (30., 3, 1.), (31., 3, 1.),
           (16., 2, 1.)],
          dtype=[('perlen', '<f8'), ('nstp', '<i8'), ('tsmult', '<f8')])

In [37]:
pd.DataFrame.from_records(perioddata)

,perlen,nstp,tsmult
0,1.0,1,1.0
1,13.0,2,1.0
2,28.0,3,1.0
3,31.0,3,1.0
4,30.0,3,1.0
5,31.0,3,1.0
6,30.0,3,1.0
7,9.0,1,1.0
8,22.0,2,1.0
9,31.0,3,1.0


### Stress Period Data

In [38]:
# boundary conditions for chem_stress
# get boundary condition packages with transport
# read in spd stress period data ... which varries based on package...

# for well package (wel):
wel = gwf.get_package('wel')
if wel.has_stress_period_data == True:
    spd_wel_dict = wel.stress_period_data.get_data(full_data=True) # full data is default
display(spd_wel_dict)

{0: rec.array([((2, 462), 0, 0.00429, 25, 0., 'krasr')],
           dtype=[('cellid', 'O'), ('q', '<i8'), ('tds', '<f8'), ('temp', '<i8'), ('cellgrp', '<f8'), ('boundname', 'O')]),
 1: rec.array([((2, 462), 668359.8449, 0.00429, 25, 0., 'krasr')],
           dtype=[('cellid', 'O'), ('q', '<f8'), ('tds', '<f8'), ('temp', '<i8'), ('cellgrp', '<f8'), ('boundname', 'O')]),
 2: rec.array([((2, 462), 668359.8449, 0.00429, 25, 0., 'krasr')],
           dtype=[('cellid', 'O'), ('q', '<f8'), ('tds', '<f8'), ('temp', '<i8'), ('cellgrp', '<f8'), ('boundname', 'O')]),
 3: rec.array([((2, 462), 668359.8449, 0.00429, 25, 0., 'krasr')],
           dtype=[('cellid', 'O'), ('q', '<f8'), ('tds', '<f8'), ('temp', '<i8'), ('cellgrp', '<f8'), ('boundname', 'O')]),
 4: rec.array([((2, 462), 668359.8449, 0.00429, 25, 0., 'krasr')],
           dtype=[('cellid', 'O'), ('q', '<f8'), ('tds', '<f8'), ('temp', '<i8'), ('cellgrp', '<f8'), ('boundname', 'O')]),
 5: rec.array([((2, 462), 668359.8449, 0.00429, 25, 0.,

In [39]:

# data stored in a dictionary of numpy record arrays
# which allows easy concatination into a single dataframe, using
# flopy dataframe interface that includes auxilary data (i.e. components) when present
spd_wel_df_dict = wel.stress_period_data.dataframe
spd_wel_df = pd.concat(spd_wel_df_dict.values(), keys=spd_wel_df_dict.keys())
spd_wel_df = spd_wel_df.droplevel(level=1)
# spd_wel_df.index.set_names(['stress_period_id'], inplace=True)
spd_wel_df.info()
spd_wel_df

<class 'pandas.core.frame.DataFrame'>
Index: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cellid_layer  21 non-null     int64  
 1   cellid_cell   21 non-null     int64  
 2   q             21 non-null     float64
 3   tds           21 non-null     float64
 4   temp          21 non-null     int64  
 5   cellgrp       21 non-null     float64
 6   boundname     21 non-null     object 
dtypes: float64(3), int64(3), object(1)
memory usage: 1.3+ KB


,cellid_layer,cellid_cell,q,tds,temp,cellgrp,boundname
0,2,462,0.000000,0.00429,25,0.0,krasr
1,2,462,668359.844900,0.00429,25,0.0,krasr
2,2,462,668359.844900,0.00429,25,0.0,krasr
3,2,462,668359.844900,0.00429,25,0.0,krasr
4,2,462,668359.844900,0.00429,25,0.0,krasr
5,2,462,668359.844900,0.00429,25,0.0,krasr
6,2,462,668359.844900,0.00429,25,0.0,krasr
7,2,462,668359.415142,0.00429,25,0.0,krasr
8,2,462,0.000000,0.00429,25,0.0,krasr
9,2,462,0.000000,0.00429,25,0.0,krasr


NOTE: `cellid` is a cell identifier tuple, and depends on the type of grid that is used for the simulation. 
- For a structured grid that uses the DIS input file, CELLID is the layer, row, and column. 
- For a grid that uses the DISV input file, CELLID is the layer and CELL2D number. 
- If the model uses the unstructured discretization (DISU) input file, CELLID is the node number for the cell.

# Get Geochemistry for Transport Models and their Initial Conditions

This first step is to create MF6 Groundwater Transport Models (GWT) for each transportable geochemical component, including setting initial conditions (IC).

This requires running an initial PHREEQC calculation from measured inputs, using utilities from the [`mf6rtm`](https://github.com/p-ortega/mf6rtm) package. 

Our workflow, similar to [`mf6rtm` example 4](https://github.com/p-ortega/mf6rtm/blob/main/benchmark/ex4.ipynb), requires these steps:
- read inputs by PHREEQC "keyword data blocks"
- convert to a dictionary
- instantiate `mup3d.{Block}` classes that contain the block's geochemical components
- set the grid size/shape for the components

### SOLUTION Block
See PHREEQC3 Manual, page 189

In [40]:
# Read Geochemical Inputs file
# for aqueous phase ("solution") components
solutions_df = pd.read_csv(solutions_filepath, index_col="component", comment = '#',)
solutions_df

,solution1_ic_molperL,solution2_bc_molperL
component,,
Ca,0.000386,1.050000e-04
Cl,0.020070,1.020000e-02
Mg,0.010000,5.030000e-03
Fe(+2),0.000001,2.000000e-07
Fe(+3),0.000000,0.000000e+00
S(6),0.000010,2.000000e-07
S(-2),0.000000,0.000000e+00
C(+4),0.000586,1.000000e-04
C(-4),0.000000,0.000000e+00


In [41]:
# convert dataframe to a Keyword Data Block dictionary
# NOTE: `mf6rtm.mup3D()` currently assigns block numbers by column, starting at 1
solutions_dict = mf6rtm.utils.solution_df_to_dict(solutions_df)

# add data to the mup3d class
solutions = mf6rtm.mup3d.Solutions(solutions_dict)
solutions.data

{'Ca': [0.000386, 0.000105],
 'Cl': [0.02007, 0.0102],
 'Mg': [0.01, 0.00503],
 'Fe(+2)': [1e-06, 2e-07],
 'Fe(+3)': [0.0, 0.0],
 'S(6)': [1e-05, 2e-07],
 'S(-2)': [0.0, 0.0],
 'C(+4)': [0.000586, 0.0001],
 'C(-4)': [0.0, 0.0],
 'pH': [9.104, 7.0],
 'pe': [-4.812, 14.5]}

In [42]:
solutions.names

['C(+4)',
 'C(-4)',
 'Ca',
 'Cl',
 'Fe(+2)',
 'Fe(+3)',
 'Mg',
 'S(-2)',
 'S(6)',
 'pH',
 'pe']

#### Assign SOLUTION Initial Conditions (IC) to all Grid Cells by Block Number

In [43]:
# mup3d currently requires a grid array with 3 dimensions
""" conc[0].shape = 
        (240, 2, 1, 80)
        ^     ^  ^  ^
        |     |  |  number of cells per layer (ncpl)
        |     |  dummy row dimension (always 1 for DISV)
        |     number of layers (nlay = 2)
        number of time steps (240)"""
# So assign dummy dimensions
nrow = 1
ncol = ncpl # should equal ncpl, but simplifying for now

In [44]:
# Assign solution block numbers to each in grid
# NOTE: at this stage of creating modflow transport models (gwt), we only want one cell per block

# start by assigning solution block 1 to all cells
grid_ic_solution_numbers = np.ones((nlay, 1, ncpl), dtype=int)

# Modify block assignments over grid, as needed

solutions.set_ic(grid_ic_solution_numbers)
solutions.ic

array([[[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]]], shape=(5, 1, 957))

#### Assign SOLUTION Boundary Conditions (BC) to all Inflows by Block Number
Using the Mup3D.ChemStress class to assign Stress Period Data (SPD)

In [45]:
# Create a well chemistry object
wellchem = mf6rtm.mup3d.ChemStress('wel')

# Assign solution block number to stress period data (spd)
# TODO: implement for multiple wells? See MF6RTM Example 5
sol_spd = [2] 
wellchem.set_spd(sol_spd)
wellchem.sol_spd

[2]

In [46]:
# Confirm that stress period data (spd) is properly assigned
for data_column_number in wellchem.sol_spd:
     solutions_list_index = data_column_number - 1
     for key, value in solutions.data.items():
        print(key, value[solutions_list_index])

Ca 0.000105
Cl 0.0102
Mg 0.00503
Fe(+2) 2e-07
Fe(+3) 0.0
S(6) 2e-07
S(-2) 0.0
C(+4) 0.0001
C(-4) 0.0
pH 7.0
pe 14.5


### EXCHANGE Block

See PHREEQC3 Manual, page 189

In [47]:
# Read Geochemical Inputs file for exchange phase components
exchange_df = pd.read_csv(exchanges_filepath, index_col="component")
exchange_df

,m0
component,
X,0.087


In [48]:
# convert dataframe to a Keyword Data Block dictionary
exchange_dict = {0:exchange_df.T.to_dict(index='component')}

# add data to the mup3d class
exchanger = mf6rtm.mup3d.ExchangePhases(exchange_dict)
exchanger.data

{0: {'X': {'m0': 0.087}}}

In [49]:
exchanger.names

['X']

#### Assign EXHANGE Initial Conditions (IC) to all Grid Cells by Block Number

In [50]:
# Set Solution Block Number for equilibration
# TODO: eliminate need for this by equilibrating to solutions blocks specied over the IC grid
exchanger.set_equilibrate_solutions([1])

# Assign block numbers to each cell
# NOTE: at this stage of creating modflow transport models (gwt), we only want one cell per block
# start by assigning exchange block 0 to all cells
grid_ic_exchange_numbers = np.ones((nlay, 1, ncpl), dtype=int)

exchanger.set_ic(grid_ic_exchange_numbers)
exchanger.ic

array([[[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 1, ..., 1, 1, 1]]], shape=(5, 1, 957))

### EQUILIBRIUM PHASES Block

See PHREEQC3 Manual, page ??

In [51]:
#equilibrium phases
equilibriums_df = pd.read_csv(equilibrium_phases_filepath)
equilibriums_df

,phase,sat_index,conc_mol_lb,num
0,Goethite,3.0,0.027,1


In [52]:
equilibriums_dict = mf6rtm.utils.parse_equilibriums_dataframe(equilibriums_df)
equilibrium_phases = mf6rtm.mup3d.EquilibriumPhases(equilibriums_dict)
equilibrium_phases.set_ic(1)
equilibrium_phases.data

{1: {'Goethite': {'si': 3.0, 'm0': 0.027}}}

### KINETICS Block

See PHREEQC3 Manual, page ??

In [53]:
#kinetics phases
kinetic_phases_df = pd.read_csv(kinetic_phases_filepath, comment = '#',)
kinetic_phases_df

,phase,m0,parm1,parm2,parm3,parm4,num
0,Calcite,4.00,100.00,0.6,NaN,NaN,1
1,Pyrite,0.04,3.42,0.0,0.5,0.0,1


In [54]:
kinetic_phases_dict = mf6rtm.utils.parse_kinetics_dataframe(kinetic_phases_df)
kinetic_phases = mf6rtm.mup3d.KineticPhases(kinetic_phases_dict)
kinetic_phases.set_ic(1)
kinetic_phases.data

{1: {'Calcite': {'m0': 4.0, 'parms': [100.0, 0.6]},
  'Pyrite': {'m0': 0.04, 'parms': [3.42, 0.0, 0.5, 0.0]}}}

### SURFACE Block

See PHREEQC3 Manual, page ??

In [55]:
#kinetics phases
surfaces_df = pd.read_csv(surfaces_filepath)
surfaces_df

,surface,site_name,switch,sites_mol,sp_area,num
0,Hfo_w,Goethite,equilibrium_phase,0.2,600.0,1


In [56]:
surfaces_dict = mf6rtm.utils.surfaces_csv_to_dict(surfaces_filepath)
surfaces = mf6rtm.mup3d.Surfaces(surfaces_dict)
surfaces.set_ic(1)
surfaces.data

{1: {'Hfo_w': ['Goethite',
   'equilibrium_phase',
   '2.000000e-001',
   '6.000000e+002']}}

### Create reaction model (RM) instance with `mf6rtm` `Mup3d` class

In [57]:
# create model class, with solution initial conditions
reaction_model = mf6rtm.mup3d.Mup3d(simulation_name, solutions, nlay, nrow, ncol)

# set model workspace for saving outputs
reaction_model.set_wd(sim_ws)

# set Phreeqc database
reaction_model.set_database(phreeqc_database_filepath)

reaction_model.set_initial_temp([7., 7., 7.])

# set chemistry domain initilization to model object
reaction_model.set_exchange_phases(exchanger)
reaction_model.set_phases(equilibrium_phases)
reaction_model.set_phases(surfaces)
# reaction_model.set_phases(kinetic_phases)

# set Phreeqc postfix file
reaction_model.set_postfix(postfix_filepath)

print(reaction_model.name, reaction_model.grid_shape)

MF6RTM Example 5 (5, 1, 957)


### Set Component H2O to transport excess H & O

[`SetComponentH2O()`](https://usgs-coupled.github.io/phreeqcrm/namespacebmiphreeqcrm.html#a0e152e5b6933e3e6bd8c79245917639a) is a PhreeqcRM function to select whether to include H2O in the component list. 
- The concentrations of H and O must be known accurately (8 to 10 significant digits) for the numerical method of PHREEQC to produce accurate pH and pe values. 
- Because most of the H and O are in the water species, it may be more robust (require less accuracy in transport) to transport the excess H and O (the H and O not in water) and water. 
- The default setting (true) is to include water, excess H, and excess O as components. 
- A setting of false will include total H and total O as components. 
- `SetComponentH2O` must be called before `FindComponents`. 

NOTE: The default for `mf6rtm` is FALSE, to use total H & O as components.

In [58]:
reaction_model.set_componenth2o(True) # True = transport H20 and excess H & O

True

### Initialize IC Chemistry over Model Grid 
This creates a PhreeqcRM instance based on components in Solution Blocks assigned initial conditions over the grid. It then runs a PHREEQC time zero equilibrium calculation for inital speciation.

In [59]:
# Intializing the mup3d class calculates the equilibrated
# initial concentration array
# NOTE: It appears that nthreads cannot be increased above 1 if using the Python phreeqcrm package
# See https://github.com/p-ortega/mf6rtm/issues/54

reaction_model.initialize(
    nthreads=4, 
    add_charge_flag=True,
)

          Estimated efficiency of chemistry without communication: 99.1079
          Cells shifted between threads     0
          Time rebalancing load             5.60284e-05


Using temperatue of 7.0 from SOLUTION 1 for all cells
MF6RTM will run with the following configuration:
  Reactive: True
  Reaction timing: all
  External files flag: False
  Emulator flag: False
  Reactions calculated at all time steps
Simulation saved in /Users/aaufdenkampe/Documents/git_mf6rtm/mf6rtm-asr-example/sims/sim03-Wallis2011/ws2x
Phreeqc initialized


In [60]:
reaction_model.phreeqc_rm.GetThreadCount()

4

In [61]:
reaction_model.components

['H2O', 'H', 'O', 'Charge', 'C', 'Ca', 'Cl', 'Fe', 'Mg', 'S']

In [62]:
# 1D array of concentrations in units of mol/L 
# structured for PhreeqcRM `GetConcentrations()` and BMI with
# component concentratinon arrays for the grid ordered as `model.components`
# Equivalent to `c_dbl_vect` (concentration double vector) in mf6rtm source code
reaction_model.init_conc_array_phreeqc

array([5.5492801e+01, 5.5492801e+01, 5.5492801e+01, ..., 9.9975830e-06,
       9.9975830e-06, 9.9975830e-06], shape=(47850,))

In [63]:
# Get component concentrations for selected grid cell
cell_index = 0
ncomps_by_nxyz_conc_array = np.reshape(
    reaction_model.init_conc_array_phreeqc, 
    (len(reaction_model.components), -1),
)
ncomps_by_nxyz_conc_array[:,cell_index]

array([5.54928010e+01, 5.02999083e-04, 1.80512394e-03, 2.27024820e-15,
       5.85858359e-04, 3.85906705e-04, 2.00651491e-02, 9.99772966e-07,
       9.99758302e-03, 9.99758300e-06])

In [64]:
# Get component concentrations for a selected grid cell
# converting to units of moles per m^3 (or mmol/L) for modflow
cell_index = 0
ic_df = pd.DataFrame(
    ncomps_by_nxyz_conc_array[:,cell_index] * 1000, # unit conversion
    index=reaction_model.components,
    columns=["initial_conc_mmolL"],
)
ic_df.index.rename("components", inplace=True)
ic_df.index = ic_df.index.astype(pd.CategoricalDtype(ordered=True))
ic_df

,initial_conc_mmolL
components,
H2O,5.549280e+04
H,5.029991e-01
O,1.805124e+00
Charge,2.270248e-12
C,5.858584e-01
Ca,3.859067e-01
Cl,2.006515e+01
Fe,9.997730e-04
Mg,9.997583e+00


In [65]:
# Dictionary of concentrations in units of moles per m^3 (or mmol/L), 
# and structured to match the shape of Modflow's grid
# reaction_model.sconc

### Initialize BC Chemistry for all Inflows

In [66]:
# Set and initialize stress period chemical concentrations for each well
reaction_model.set_chem_stress(wellchem)

Initializing ChemStress
ChemStress wel initialized


In [67]:
# Component names
reaction_model.wel.auxiliary

['H2O', 'H', 'O', 'Charge', 'C', 'Ca', 'Cl', 'Fe', 'Mg', 'S']

In [68]:
# Equilbrated concentrations Well 0 boundary conditions (from Solution 2)
# in units of moles per m^3 (or mmol/L)
reaction_model.wel.data

{0: [np.float64(55496.790769982064),
  np.float64(0.0702538771122363),
  np.float64(0.2709615928893072),
  np.float64(1.4325623424618673e-08),
  np.float64(0.09998301825469404),
  np.float64(0.1049821691637596),
  np.float64(10.198267861622368),
  np.float64(0.00019996603700090428),
  np.float64(5.029145825199502),
  np.float64(0.00019996603657224593)]}

In [69]:
# Open data for a specifc well as a dataframe
well_id = 0
bc_df = pd.DataFrame(
    reaction_model.wel.data[well_id],
    index=reaction_model.wel.auxiliary,
    columns=["initial_conc_mmolL"],
)
bc_df.index.rename("components", inplace=True)
bc_df.index = ic_df.index.astype(pd.CategoricalDtype(ordered=True))
bc_df

,initial_conc_mmolL
components,
H2O,5.549679e+04
H,7.025388e-02
O,2.709616e-01
Charge,1.432562e-08
C,9.998302e-02
Ca,1.049822e-01
Cl,1.019827e+01
Fe,1.999660e-04
Mg,5.029146e+00


In [70]:
# Get Modflow's Stress Period Data (spd) from the `wel` package,
# with the well location (cellid), flow rate (q), and other conditions
# as previously collected above
spd_wel_df

,cellid_layer,cellid_cell,q,tds,temp,cellgrp,boundname
0,2,462,0.000000,0.00429,25,0.0,krasr
1,2,462,668359.844900,0.00429,25,0.0,krasr
2,2,462,668359.844900,0.00429,25,0.0,krasr
3,2,462,668359.844900,0.00429,25,0.0,krasr
4,2,462,668359.844900,0.00429,25,0.0,krasr
5,2,462,668359.844900,0.00429,25,0.0,krasr
6,2,462,668359.844900,0.00429,25,0.0,krasr
7,2,462,668359.415142,0.00429,25,0.0,krasr
8,2,462,0.000000,0.00429,25,0.0,krasr
9,2,462,0.000000,0.00429,25,0.0,krasr


In [71]:
# # Append Conc data to Well Stress Period Data list, 
# # NOTE: only run this once
# for i in range(len(wel_spd)):
#     wel_spd[i].extend(reaction_model.wel.data[i])
# wel_spd

### Unit Conversions

- Although MODFLOW is technically agnostic about chemical concentration units used for transport, we have found solver issues when units between transport and reaction models are different.

#### PHREEQC unit handling
- Although PHREEQC can handle multiple units, all options use the metric system. From PHREEQC3 Manual page 191: 
  - Three groups of concentration units are allowed, concentration 
    - (1) per liter (“/L”), 
    - (2) per kilogram solution (“/kgs”), or 
    - (3) per kilogram water (“/kgw”). 
  - All concentration units for a solution must be within the same group. 
  - Within a group, either grams or moles may be used, and prefixes milli (m) and micro (u) are acceptable. The abbreviations for parts per thousand, “ppt”; parts per million, “ppm”; and parts per billion, “ppb”, are acceptable in the “per kilogram solution” group. 
  - Default is mmol/kgw.

#### PhreeqcRM unit defaults
- [`YAMLSetUnitsSolution()`](https://usgs-coupled.github.io/phreeqcrm/namespaceyamlphreeqcrm.html#a6ae20ea754c0f1087ba700dbf48b55a4) uses:
  - 1, mg/L (default); 
  - 2 mol/L; or 
  - 3, mass fraction, kg/kgs.

# Add Chem to Modflow 

## Create MF6 Transport Models for each chemical component
With initial starting concentrations calculated from initializing PhreeqcRM via the `mup3d.sconc` dictionary, with units of of moles per m^3 (or mmol/L).

In [72]:
component_name_l = reaction_model.sconc.keys()
component_name_l

dict_keys(['H2O', 'H', 'O', 'Charge', 'C', 'Ca', 'Cl', 'Fe', 'Mg', 'S'])

In [73]:
reaction_model.sconc['Ca']

array([[[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
         0.3859067]],

       [[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
         0.3859067]],

       [[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
         0.3859067]],

       [[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
         0.3859067]],

       [[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
         0.3859067]]], shape=(5, 1, 957))

In [74]:
# create new gwt models for each component
porosity = 0.3
dispersivity = 0.00656  # ft = 0.002 # Longitudinal dispersivity (m)
gwf_name = "flow"

for component_name in component_name_l:
    print("Adding gwt model for: " + component_name)
    gwt_name = "trans-" + component_name
    sim = utils.create_mf6_gwt(
        sim, gwf_name, gwt_name, component_name, 
        reaction_model.sconc[component_name],
        porosity, dispersivity
    )

Adding gwt model for: H2O
Adding gwt model for: H
Adding gwt model for: O
Adding gwt model for: Charge
Adding gwt model for: C
Adding gwt model for: Ca
Adding gwt model for: Cl
Adding gwt model for: Fe
Adding gwt model for: Mg
Adding gwt model for: S


In [75]:
# Confirm Modflow models in the simulation
sim.model_names

['flow',
 'trans-tds',
 'trans-temp',
 'trans-H2O',
 'trans-H',
 'trans-O',
 'trans-Charge',
 'trans-C',
 'trans-Ca',
 'trans-Cl',
 'trans-Fe',
 'trans-Mg',
 'trans-S']

In [76]:
# Confirm initial condition concs for Na, from `mup3d.sconc`
# units of moles per m^3 (or mmol/L), 
sim.get_model('trans-Ca').ic.strt.array

array([[0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
        0.3859067],
       [0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
        0.3859067],
       [0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
        0.3859067],
       [0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
        0.3859067],
       [0.3859067, 0.3859067, 0.3859067, ..., 0.3859067, 0.3859067,
        0.3859067]], shape=(5, 957))

## Add Chem Components to Stress Period Data

In [77]:
# We created this dataframe from mf6rtm.mup3d inputs
bc_df

,initial_conc_mmolL
components,
H2O,5.549679e+04
H,7.025388e-02
O,2.709616e-01
Charge,1.432562e-08
C,9.998302e-02
Ca,1.049822e-01
Cl,1.019827e+01
Fe,1.999660e-04
Mg,5.029146e+00


In [78]:
# make aliases for well component names and concentrations
# units of moles per m^3 (or mmol/L), 
component_name_l = reaction_model.wel.auxiliary
wel_conc = reaction_model.wel.data[0]
display(component_name_l, wel_conc)

['H2O', 'H', 'O', 'Charge', 'C', 'Ca', 'Cl', 'Fe', 'Mg', 'S']

[np.float64(55496.790769982064),
 np.float64(0.0702538771122363),
 np.float64(0.2709615928893072),
 np.float64(1.4325623424618673e-08),
 np.float64(0.09998301825469404),
 np.float64(0.1049821691637596),
 np.float64(10.198267861622368),
 np.float64(0.00019996603700090428),
 np.float64(5.029145825199502),
 np.float64(0.00019996603657224593)]

In [79]:
# add new components to wel spd and auxvar

# load wel package and stress period data
wel = gwf.wel
spd = wel.stress_period_data.get_data(full_data=True) 
    # NOTE: alsp defined above as `spd_wel_dict`

# modify wel spd data
new_wel_spd = {}
for kper, records in spd.items():
    updated_record = utils.modify_wel_spd(records, component_name_l, wel_conc)
    new_wel_spd[kper] = np.rec.array(updated_record)

# set new aux variables
wel_spd_dtype = list(new_wel_spd[0].dtype.names)
new_wel_auxvar = wel_spd_dtype[2:-1]  # "2:-1" --> excludes wel parameters from auxvars
wel.auxiliary = new_wel_auxvar

# set stress period data to new_wel_spd that includes added components
wel.stress_period_data.set_data(new_wel_spd)

In [80]:
# Confirm well concentrations, units of moles per m^3 (or mmol/L)
wel.stress_period_data.dataframe[0]

,cellid_layer,cellid_cell,q,tds,temp,cellgrp,H2O,H,O,Charge,C,Ca,Cl,Fe,Mg,S,boundname
0,2,462,0,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr


In [81]:
# Confirm well concentrations, units of moles per m^3 (or mmol/L)
# for every stress period
spd_welchem_df_dict = wel.stress_period_data.dataframe
spd_welchem_df = pd.concat(spd_welchem_df_dict.values(), keys=spd_welchem_df_dict.keys())
spd_welchem_df = spd_welchem_df.droplevel(level=1)
spd_welchem_df

,cellid_layer,cellid_cell,q,tds,temp,cellgrp,H2O,H,O,Charge,C,Ca,Cl,Fe,Mg,S,boundname
0,2,462,0.000000,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
1,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
2,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
3,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
4,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
5,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
6,2,462,668359.844900,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
7,2,462,668359.415142,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
8,2,462,0.000000,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr
9,2,462,0.000000,0.00429,25,0.0,55496.79077,0.070254,0.270962,1.432562e-08,0.099983,0.104982,10.198268,0.0002,5.029146,0.0002,krasr


## Modify timestep length

In [82]:
# modify tdis to change timestep length and total simulation time
change_nstp = True
    # if False, timestep lenght varies by stess period
nstp_multiplier = 2
number_of_days_last_stressperiod = 300.

if change_nstp == True:
    tdis_spd = sim.get_package("tdis").perioddata.get_data(full_data=True)
    tdis_spd["nstp"] = tdis_spd["nstp"] * nstp_multiplier
    # Modify length of last stress period
    tdis_spd['perlen'][-1] = number_of_days_last_stressperiod
    tdis_spd['nstp'][-1] = int(number_of_days_last_stressperiod 
                               / (nstp_multiplier * 5)) 
                            # if 5, then double length of other periods
    sim.get_package("tdis").perioddata.set_data(tdis_spd)

In [83]:
tdis_spd_df = pd.DataFrame.from_records(tdis.perioddata.get_data())
tdis_spd_df['stp_len'] = tdis_spd_df['perlen'] / tdis_spd_df['nstp']
tdis_spd_df

,perlen,nstp,tsmult,stp_len
0,1.0,2,1.0,0.500000
1,13.0,4,1.0,3.250000
2,28.0,6,1.0,4.666667
3,31.0,6,1.0,5.166667
4,30.0,6,1.0,5.000000
5,31.0,6,1.0,5.166667
6,30.0,6,1.0,5.000000
7,9.0,2,1.0,4.500000
8,22.0,4,1.0,5.500000
9,31.0,6,1.0,5.166667


## Remove transport models that are not needed

In [84]:
# Remove transport models for testing
sim.remove_model('trans-tds')
sim.remove_model('trans-temp')

In [85]:
gwt_model_names = [name for name in sim.model_names 
                    if (sim.get_model(name).model_type == 'gwt6')]
print("Number of transport models: ",len(gwt_model_names))
gwt_model_names

Number of transport models:  10


['trans-H2O',
 'trans-H',
 'trans-O',
 'trans-Charge',
 'trans-C',
 'trans-Ca',
 'trans-Cl',
 'trans-Fe',
 'trans-Mg',
 'trans-S']

In [86]:
# write updated simulation input files
sim.write_simulation()

# Run Modflow 6 simulation only

To confirm that conservative transport is occuring as expected.

In [87]:
utils.run_models(sim, silent=False)

FloPy is using the following executable to run the model: ../../../bin/mf6.7.0/macarm/mf6
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.7.0 02/05/2026

        MODFLOW 6 compiled Feb  5 2026 22:36:20 with GCC version 13.4.0

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warranty, expressed or 
implied, is made by the USGS or the U.S. Government as to the 
functionality of the software and related material nor shall the 
fact of release constitute any such warranty. Furthermore, the 
software is released on condition that neither the USGS nor the U.S. 
Government shall be held liable for any damages resulting from its 
authorized or unauthorized use. Also refer to the USGS Water 
Resour

## Plot MF6 Transport Results with no Reactions

When just running MF6, before any coupling.

For addtional result plots of the simple ASR Modflow 6 simulation used throughout this repository, see `sims/sim00-mf6only/mf6_explore.ipynb`

In [88]:
# Create list of components to plot based on intersection with transported components
components_to_plot = [c for c in component_name_l if c in ['Ca', 'Cl', 'K', 'N', 'Na']]
components_to_plot

['Ca', 'Cl']

# Reactive Transport Simulation
Using MF6RTM

In [89]:
# Does this cell run?
"Yes"

'Yes'

In [90]:
# Run the model using this wrapper function for `mf6rtm.solve(model.wd)`
reaction_model.run()

Running mf6rtm
Using libmf6 found in model directory: libmf6.dylib
Processing initial chemistry configuration
Starting Solution at 2026-03-27 15:28:43
Transport       | Stress period:  1     | Time step:      1          | Completed in :  0 min   3.92e-05 sec
Reactions       | Stress period:  1     | Time step:      1          | Completed in :  0 min   8.92e-03 sec
Transport       | Stress period:  1     | Time step:      2          | Completed in :  0 min   3.91e-05 sec
Reactions       | Stress period:  1     | Time step:      2          | Completed in :  0 min   9.04e-03 sec
Transport       | Stress period:  2     | Time step:      1          | Completed in :  0 min   4.75e-05 sec
Reactions       | Stress period:  2     | Time step:      1          | Completed in :  0 min   9.04e-03 sec
Transport       | Stress period:  2     | Time step:      2          | Completed in :  0 min   4.54e-05 sec
Reactions       | Stress period:  2     | Time step:      2          | Completed in :  0 min 

True

Using MF6RTM Chemistry
- Runs with SOLUTIONS, EXCHANGES
- Crashes with SOLUTIONS, EXCHANGES, EQUILIBRIUM PHASES, SURFACE, KINETICS
  - C(4) concs negative after 2nd timestep
  - Tried modifying chemistry inputs, to get more reasonable pH, ALK, DIC, and charge balance but still crashed
- 1.5125 mins with all but KINETICS
- 1.5717 mins all, including KINETICS, but removing calcite
- 1.6881 mins before `refactor/cdbl_vect`

## Visualize MF6RTM Results

In [91]:
# read in mf6 conc results
sim_rxn = flopy.mf6.MFSimulation.load(
    sim_ws=sim_ws,
    exe_name=mf6_exe,  #'mf6',
    verbosity_level=0,
)
conc_rxn = utils.get_concentrations(sim_rxn, component_name_l)
times_c_rxn = utils.get_times_c(sim_rxn, component_name_l)

In [92]:
# read in phreeqc results 
sout_df = pd.read_csv(
    sim_ws / 'sout.csv', 
    sep = ',', 
    skipinitialspace=True, 
    index_col=[0],
)
sout_df.info()
sout_df

<class 'pandas.core.frame.DataFrame'>
Index: 641190 entries, 0.5 to 809.0
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   pH              641190 non-null  float64
 1   pe              641190 non-null  float64
 2   Alk(eq/kgw)     641190 non-null  float64
 3   Ca(mol/kgw)     641190 non-null  float64
 4   Mg(mol/kgw)     641190 non-null  float64
 5   Cl(mol/kgw)     641190 non-null  float64
 6   S(6)(mol/kgw)   641190 non-null  float64
 7   C(4)(mol/kgw)   641190 non-null  float64
 8   S(-2)(mol/kgw)  641190 non-null  float64
 9   Fe(2)(mol/kgw)  641190 non-null  float64
 10  Fe(3)(mol/kgw)  641190 non-null  float64
 11  cell            641190 non-null  float64
dtypes: float64(12)
memory usage: 63.6 MB


,pH,pe,Alk(eq/kgw),Ca(mol/kgw),Mg(mol/kgw),Cl(mol/kgw),S(6)(mol/kgw),C(4)(mol/kgw),S(-2)(mol/kgw),Fe(2)(mol/kgw),Fe(3)(mol/kgw),cell
time,,,,,,,,,,,,
0.5,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319892e-13,1.000000e-06,1.464024e-11,0.0
0.5,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319892e-13,1.000000e-06,1.464024e-11,1.0
0.5,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319892e-13,1.000000e-06,1.464024e-11,2.0
0.5,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319892e-13,1.000000e-06,1.464024e-11,3.0
0.5,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319892e-13,1.000000e-06,1.464024e-11,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...
809.0,9.099752,-4.768139,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319640e-13,9.999990e-07,1.464019e-11,4780.0
809.0,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319888e-13,1.000000e-06,1.464023e-11,4781.0
809.0,9.099755,-4.768148,0.000684,0.000386,0.01,0.02007,0.00001,0.000586,2.319889e-13,1.000000e-06,1.464023e-11,4782.0


## Holoviz Plots

In [93]:
import hvplot.pandas
import holoviews as hv

In [94]:
wel_cellid

(2, 462)

In [95]:
# Plot one cell away from well
cell_to_plot = (wel_cellid[0], wel_cellid[1]+1)
cell_to_plot

(2, 463)

In [96]:
cell_flatid = cellid_flatmap[*cell_to_plot]
cell_flatid

np.int64(2377)

In [97]:
plot_df = sout_df.loc[sout_df.cell == cell_flatid]
plot_df.columns

Index(['pH', 'pe', 'Alk(eq/kgw)', 'Ca(mol/kgw)', 'Mg(mol/kgw)', 'Cl(mol/kgw)',
       'S(6)(mol/kgw)', 'C(4)(mol/kgw)', 'S(-2)(mol/kgw)', 'Fe(2)(mol/kgw)',
       'Fe(3)(mol/kgw)', 'cell'],
      dtype='object')

In [98]:
# Create list of components to plot based on intersection with transported components
components_to_plot = [f'{c}' for c in component_name_l if c in ['Ca', 'Cl', 'K', 'N', 'Na',]]
components_to_plot.append('S(6)')
components_to_plot

['Ca', 'Cl', 'S(6)']

In [99]:
major_elements = ['Ca(mol/kgw)', 'Mg(mol/kgw)', 'Cl(mol/kgw)',
       'S(6)(mol/kgw)', 'C(4)(mol/kgw)',]
majors_plot = plot_df[major_elements].hvplot(ylabel='Concentrations (mol/kgw)', logy=False)

In [100]:
minor_elements = ['S(-2)(mol/kgw)', 'Fe(2)(mol/kgw)',
       'Fe(3)(mol/kgw)',]
minors_plot = plot_df[minor_elements].hvplot(ylabel='Concentrations (mol/kgw)', logy=True)

In [101]:
ph_plot = plot_df[['pH']].hvplot(ylabel='pH')
pe_plot = plot_df[['pe']].hvplot(ylabel='pe')

In [102]:
plot_list = [majors_plot, minors_plot, ph_plot, pe_plot]
hv.Layout(plot_list).cols(1).opts(
    title=f'MF6RTM Results for "{simulation_name}" at cellid {cell_to_plot}',
    shared_axes=False, 
    axiswise=True,
)

:Layout
   .NdOverlay.I  :NdOverlay   [Variable]
      :Curve   [time]   (value)
   .NdOverlay.II :NdOverlay   [Variable]
      :Curve   [time]   (value)
   .Curve.I      :Curve   [time]   (pH)
   .Curve.II     :Curve   [time]   (pe)

# END